In [1]:
import numpy as np
import pandas as pd
import matplotlib as plt
# from sklearn.feature_extraction.text import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform
from sklearn.metrics import accuracy_score
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import joblib
# import re as regex
import string

In [2]:
def remove_stopwords_and_specials(text):
    stop_words = set(stopwords.words('english'))
    words_tokens = word_tokenize(text)
    
    # creates regex of special characters and punctuation
    # special_chars = regex.compile('[@_!#$%^&*()<>?/\|}{~:]')
    punc = string.punctuation

    filtered_sentence = [w for w in words_tokens if not w in stop_words and w not in punc]

    return ' '.join(filtered_sentence)

In [3]:
def clean_data(text):
    # removes stop words
    no_stops = remove_stopwords_and_specials(text.lower())
    


In [4]:
def create_sentiment_integer(sentiment):
    if sentiment == 'negative':
        return -1
    elif sentiment == 'positive':
        return 1
    return 0

In [23]:
df = pd.read_csv('../data/LargeTweetsDataset.csv')
df = df.iloc[::2]
df.head()

,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew
6,0,1467811594,Mon Apr 06 22:20:03 PDT 2009,NO_QUERY,coZZ,@LOLTrish hey long time no see! Yes.. Rains a...
8,0,1467812025,Mon Apr 06 22:20:09 PDT 2009,NO_QUERY,mimismo,@twittera que me muera ?


In [24]:
len(df)

800000

In [25]:
df.columns

Index(['0', '1467810369', 'Mon Apr 06 22:19:45 PDT 2009', 'NO_QUERY',
       '_TheSpecialOne_',
       '@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D'],
      dtype='object')

In [26]:
df = df.drop(['1467810369', 'NO_QUERY', '_TheSpecialOne_', 'Mon Apr 06 22:19:45 PDT 2009'], axis=1)
df.columns = ['Polarity', 'Tweet']

In [27]:
df.head()

,Polarity,Tweet
0,0,is upset that he can't update his Facebook by ...
2,0,my whole body feels itchy and like its on fire
4,0,@Kwesidei not the whole crew
6,0,@LOLTrish hey long time no see! Yes.. Rains a...
8,0,@twittera que me muera ?


In [28]:
len(df)

800000

In [29]:
# cleans data of stop words, special characters, and punctuation
df['Tweet'] = df['Tweet'].apply(str).apply(remove_stopwords_and_specials)
df

,Polarity,Tweet
0,0,upset ca n't update Facebook texting ... might...
2,0,whole body feels itchy like fire
4,0,Kwesidei whole crew
6,0,LOLTrish hey long time see Yes .. Rains bit bi...
8,0,twittera que muera
...,...,...
1599990,4,rmedina LaTati Mmmm That sounds absolutely per...
1599992,4,SCOOBY_GRITBOYS
1599994,4,Just woke Having school best feeling ever
1599996,4,Are ready MoJo Makeover Ask details


In [30]:
tfidf = TfidfVectorizer(strip_accents=None, lowercase=False, preprocessor=None)
X = tfidf.fit_transform(df['Tweet'].values.astype('U'))

In [31]:
y = df['Polarity']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1)
params_per_model =[
    {
        'SupportVectorClass': {
            'C': uniform(0.1, 10),
            'kernel': ['linear', 'poly', 'rbf'],
            'gamma': ['scale', 'auto']
        }
    }
]

In [32]:
# { 'C': 1.7, 'max_iter': 200, 'penalty': 'l1', 'solver': 'saga'}
lgr = LogisticRegression()
lgr.set_params(C=1.7, max_iter=200, penalty='l1', solver='saga')


LogisticRegression(C=1.7, max_iter=200, penalty='l1', solver='saga')

In [33]:
lgr.fit(X_train, y_train)

C:\Users\elija\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


LogisticRegression(C=1.7, max_iter=200, penalty='l1', solver='saga')

In [34]:
# predictions
preds_log = lgr.predict(X_test)

In [35]:
# accuracy scores
print(f"Train Score: {lgr.score(X_train, y_train)}")
print(f"Test Score: {lgr.score(X_test, y_test)}")

Train Score: 0.8110958333333333
Test Score: 0.7840875


In [36]:
joblib.dump(lgr, '../models/large_logistic_regression_model.pkl')

['../models/large_logistic_regression_model.pkl']